# Data Warehouse Quality Assurance (QA) Framework

An automated test suite for validating data integrity across Data Warehouse (DWH) pipeline layers.

**Supported layers:** DIM · XREF · STG · HZ · DWH

**Tests included:**
1. Null check on surrogate keys
2. Duplicate check on DIM table
3. Duplicate check on XREF table
4. Surrogate key from DIM present in XREF
5. Source key from STG present in XREF
6. Historical date period overlap check (DWH)
7. Data diff between STG and DIM
8. Row count reconciliation between HZ and DWH

In [ ]:
# Install required ODBC driver for SQL Server connectivity
pip install pyodbc

In [ ]:
import pyodbc
import signal
import ipywidgets as widgets
from IPython.display import display, Javascript
from IPython.core.debugger import Pdb

## Step 1: Enter Database Connection & Table Details

Fill in the fields below, then click **Run All Cells Below**.

In [ ]:
# Input widgets for database connection and table names
serverName        = widgets.Text(value='', placeholder='Server Name')
databaseName      = widgets.Text(value='', placeholder='Database Name')

dim_schema_name_wid  = widgets.Text(value='', placeholder='DIM Schema Name')
dim_table_name_wid   = widgets.Text(value='', placeholder='DIM Table Name')

xref_schema_name_wid = widgets.Text(value='', placeholder='XREF Schema Name')
xref_table_name_wid  = widgets.Text(value='', placeholder='XREF Table Name')

stg_schema_name_wid  = widgets.Text(value='', placeholder='STG Schema Name')
stg_table_name_wid   = widgets.Text(value='', placeholder='STG Table Name')

hz_schema_name_wid   = widgets.Text(value='', placeholder='HZ Schema Name')
hz_table_name_wid    = widgets.Text(value='', placeholder='HZ Table Name')

dwh_schema_name_wid  = widgets.Text(value='', placeholder='DWH Schema Name')
dwh_table_name_wid   = widgets.Text(value='', placeholder='DWH Table Name')

In [ ]:
widgets.GridBox(
    children=[
        serverName, databaseName,
        dim_schema_name_wid, dim_table_name_wid,
        xref_schema_name_wid, xref_table_name_wid,
        stg_schema_name_wid, stg_table_name_wid,
        hz_schema_name_wid, hz_table_name_wid,
        dwh_schema_name_wid, dwh_table_name_wid
    ],
    layout=widgets.Layout(
        width='100%',
        grid_template_rows='auto auto auto',
        grid_template_columns='50% 50%',
        grid_template_areas='''
        "serverName databaseName"
        "dim_schema_name_wid dim_table_name_wid"
        "xref_schema_name_wid xref_table_name_wid"
        "stg_schema_name_wid stg_table_name_wid"
        "hz_schema_name_wid hz_table_name_wid"
        "dwh_schema_name_wid dwh_table_name_wid"
        '''
    )
)

In [ ]:
def run_all_cells_below():
    """Trigger execution of all cells below the current one."""
    display(Javascript('''
        var cells = IPython.notebook.get_cells();
        var current_cell_idx = IPython.notebook.get_selected_index();
        for (var i = current_cell_idx + 1; i < cells.length; i++) {
            cells[i].execute();
        }
    '''))

button = widgets.Button(description='Run All Cells Below')
button.on_click(lambda _: run_all_cells_below())
display(button)

In [ ]:
def stop_kernel():
    """Interrupt the kernel — used to halt execution when inputs are missing."""
    Pdb().set_trace()
    import os
    os.kill(os.getpid(), signal.SIGINT)

# Guard: ensure DB details have been entered before running tests
if dim_schema_name_wid.value == '':
    print('Database details not provided. Please fill in the widgets above and re-run.')
    stop_kernel()

## Step 2: Helper Functions

In [ ]:
def try_cast(column_list):
    """Wrap each column name in TRY_CAST(...AS NVARCHAR) for safe string comparison."""
    return [f'TRY_CAST({col} AS NVARCHAR)' for col in column_list]  # Fixed: was 'coloumnList' (typo)

## Step 3: Read Database Details

In [ ]:
# Read values from widgets into variables
server_name   = serverName.value if serverName.value != '' else 'localhost'
database_name = databaseName.value

dim_schema_name  = dim_schema_name_wid.value
dim_table_name   = dim_table_name_wid.value

xref_schema_name = xref_schema_name_wid.value
xref_table_name  = xref_table_name_wid.value

stg_schema_name  = stg_schema_name_wid.value
stg_table_name   = stg_table_name_wid.value

hz_schema_name   = hz_schema_name_wid.value
hz_table_name    = hz_table_name_wid.value

dwh_schema_name  = dwh_schema_name_wid.value
dwh_table_name   = dwh_table_name_wid.value

## Step 4: Configure Test Columns

> **Edit these lists** to match the columns in your tables before running tests.

In [ ]:
# ─── Column Configuration ──────────────────────────────────────────────────
# Columns used for duplicate checks and diff comparisons
dim_column_list  = ['column_1', 'column_2']   # Used in Test 1 (DIM duplicates) and Test 7 (STG-DIM diff)
xref_column_list = ['xref_key_column']         # Used in Test 2 (XREF duplicates)
stg_column_list  = ['stg_column_1', 'stg_column_2']  # Paired with dim_column_list for Test 7
dwh_column_list  = ['dwh_column_1', 'dwh_column_2']  # Informational; used if extended
hz_column_list   = []                           # HZ columns for count filter

# Surrogate/business key names
sk_dim_name  = 'dim_surrogate_key'   # Surrogate key in DIM table (Tests 0-null, 3-SK check)
sk_stg_name  = 'stg_source_key'      # Source key in STG table (Test 4)
businesskey  = 'business_key_column' # Business key for historical date check (Test 5)

## Step 5: Connect to Database

In [ ]:
# Initialise test result trackers
passed_list = []
failed_list = []

# Connect using Windows Trusted Authentication (adjust connection string for SQL auth if needed)
conn_string = (
    f'DRIVER=ODBC Driver 17 For SQL Server;'
    f'Server={server_name};'
    f'DATABASE={database_name};'
    f'Trusted_Connection=yes'
)
conn   = pyodbc.connect(conn_string)
cursor = conn.cursor()
print(f'Connected to {server_name} / {database_name}')

In [ ]:
# Smoke test: verify the DIM table is reachable
sql = f"SELECT TOP 10 {','.join(dim_column_list)} FROM {dim_schema_name}.{dim_table_name}"
cursor.execute(sql)
rows = cursor.fetchall()
print('Connection OK. Sample rows:', rows)

## Test 0 — Null Check on Surrogate Key

In [ ]:
def null_check(schema_name, table_name, column_name):
    """Return rows where the given column is NULL."""
    query = f'''
        SELECT *
        FROM {schema_name}.{table_name}
        WHERE {column_name} IS NULL
    '''
    cursor.execute(query)
    return cursor.fetchall()

try:
    assert null_check(dim_schema_name, dim_table_name, sk_dim_name) == []
    print(f'PASS  Null Check — {dim_schema_name}.{dim_table_name}')
    passed_list.append(f'Null Check — {dim_schema_name}.{dim_table_name}')
except AssertionError:
    print(f'FAIL  Null Check — {dim_schema_name}.{dim_table_name}')
    failed_list.append(f'Null Check — {dim_schema_name}.{dim_table_name}')

## Test 1 — Duplicate Check on DIM Table

In [ ]:
def check_duplicates_dim():
    """Return duplicate rows (by business key) in the DIM table."""
    query = f'''
        SELECT TOP 10 {','.join(try_cast(dim_column_list))}
        FROM {dim_schema_name}.{dim_table_name}
        GROUP BY {','.join(dim_column_list)}
        HAVING COUNT(1) > 1
    '''
    cursor.execute(query)
    return cursor.fetchall()

try:
    assert check_duplicates_dim() == []
    print(f'PASS  Duplicate Check DIM — {dim_schema_name}.{dim_table_name}')
    passed_list.append(f'Duplicate Check DIM — {dim_schema_name}.{dim_table_name}')
except Exception:
    print(f'FAIL  Duplicate Check DIM — {dim_schema_name}.{dim_table_name}')
    failed_list.append(f'Duplicate Check DIM — {dim_schema_name}.{dim_table_name}')

## Test 2 — Duplicate Check on XREF Table

In [ ]:
def check_duplicates_xref():
    """Return duplicate rows (by key) in the XREF table."""
    query = f'''
        SELECT TOP 10 {','.join(try_cast(xref_column_list))}, COUNT(1) AS cnt
        FROM {xref_schema_name}.{xref_table_name}
        GROUP BY {','.join(xref_column_list)}
        HAVING COUNT(1) > 1
    '''
    cursor.execute(query)
    return cursor.fetchall()

try:
    assert check_duplicates_xref() == []
    print(f'PASS  Duplicate Check XREF — {xref_schema_name}.{xref_table_name}')
    passed_list.append(f'Duplicate Check XREF — {xref_schema_name}.{xref_table_name}')
except Exception:
    print(f'FAIL  Duplicate Check XREF — {xref_schema_name}.{xref_table_name}')
    failed_list.append(f'Duplicate Check XREF — {xref_schema_name}.{xref_table_name}')

## Test 3 — Every SK from DIM Exists in XREF

Verifies referential integrity: no DIM surrogate key should be orphaned in the XREF table.

In [ ]:
def check_sk(left_schema, left_table, right_schema, right_table, sk_name):
    """
    Return rows from the left table whose surrogate key has no match in the right table.
    Uses a LEFT JOIN and filters on NULL to find orphaned keys.
    """
    query = f'''
        SELECT TOP 10 *
        FROM {left_schema}.{left_table} AS dim
        LEFT JOIN {right_schema}.{right_table} AS xref
            ON dim.{sk_name} = xref.{sk_name}
        WHERE xref.{sk_name} IS NULL
    '''
    cursor.execute(query)
    return cursor.fetchall()

try:
    rows = check_sk(dim_schema_name, dim_table_name, xref_schema_name, xref_table_name, sk_dim_name)
    assert rows == []
    print(f'PASS  SK from DIM in XREF — {dim_schema_name}.{dim_table_name} → {xref_schema_name}.{xref_table_name}')
    passed_list.append(f'SK DIM→XREF — {dim_schema_name}.{dim_table_name}')
except Exception:
    print(f'FAIL  SK from DIM in XREF — {dim_schema_name}.{dim_table_name} → {xref_schema_name}.{xref_table_name}')
    if rows:
        print('Sample orphaned rows:', rows)
    failed_list.append(f'SK DIM→XREF — {dim_schema_name}.{dim_table_name}')

## Test 4 — Every Source Key from STG Exists in XREF

Ensures all staging source keys have been mapped into the cross-reference table.

In [ ]:
try:
    rows = check_sk(stg_schema_name, stg_table_name, xref_schema_name, xref_table_name, sk_stg_name)
    assert rows == []
    print(f'PASS  SK from STG in XREF — {stg_schema_name}.{stg_table_name} → {xref_schema_name}.{xref_table_name}')
    passed_list.append(f'SK STG→XREF — {stg_schema_name}.{stg_table_name}')
except Exception:
    print(f'FAIL  SK from STG in XREF — {stg_schema_name}.{stg_table_name} → {xref_schema_name}.{xref_table_name}')
    if rows:
        print('Sample unmatched rows:', rows)
    failed_list.append(f'SK STG→XREF — {stg_schema_name}.{stg_table_name}')

## Test 5 — Historical Date Period Overlap Check (DWH)

Checks that no two records for the same business key have overlapping `period_start_ts` values.
Uses a LAG window function for efficient detection.

In [ ]:
def dwh_historical_date_check_using_lag():
    """
    Detect overlapping date ranges in a DWH historical (type-2 SCD) table.
    Returns each row labelled as 'First row', 'Overlapping date range', or 'No overlap'.
    """
    query = f'''
        SELECT
            {businesskey},
            [period_start_ts],
            CASE
                WHEN LAG([period_start_ts]) OVER (PARTITION BY {businesskey} ORDER BY [period_start_ts]) IS NULL
                    THEN 'First row'
                WHEN LAG([period_start_ts]) OVER (PARTITION BY {businesskey} ORDER BY [period_start_ts]) >= [period_start_ts]
                    THEN 'Overlapping date range'
                ELSE 'No overlap'
            END AS date_range_check
        FROM {dwh_schema_name}.{dwh_table_name}
        ORDER BY {businesskey}, [period_start_ts]
    '''
    cursor.execute(query)
    return cursor.fetchall()

try:
    results = dwh_historical_date_check_using_lag()
    overlaps = [r for r in results if r[2] == 'Overlapping date range']
    assert overlaps == []
    print(f'PASS  Date Period Overlap Check — {dwh_schema_name}.{dwh_table_name}')
    passed_list.append(f'Date Overlap Check — {dwh_schema_name}.{dwh_table_name}')
except Exception:
    print(f'FAIL  Date Period Overlap Check — {dwh_schema_name}.{dwh_table_name}')
    failed_list.append(f'Date Overlap Check — {dwh_schema_name}.{dwh_table_name}')

## Test 6 — Data Diff Between STG and DIM

Checks that all records in the STG table are present in the DIM table (no missing rows).

> **Preconditions:** STG table must not be empty. Skip for Attunity-loaded tables.

In [ ]:
def check_diff_btw_stg_dim():
    """
    Return rows present in STG but missing from DIM.
    Column pairs in stg_column_list / dim_column_list are matched positionally.
    """
    join_conditions = [
        f'AND mT.{stg_col} = sT.{dim_col}'
        for stg_col, dim_col in zip(stg_column_list, dim_column_list)
    ]
    query = f'''
        SELECT TOP 10 *
        FROM {stg_schema_name}.{stg_table_name} AS mT
        WHERE NOT EXISTS (
            SELECT 1
            FROM {dim_schema_name}.{dim_table_name} AS sT
            WHERE mT.src_sys_id = sT.src_sys_id
            {' '.join(join_conditions)}
        )
    '''
    cursor.execute(query)
    return cursor.fetchall()

try:
    assert check_diff_btw_stg_dim() == []
    print('PASS  STG vs DIM Diff Check')
    passed_list.append('STG vs DIM Diff Check')
except Exception:
    print('FAIL  STG vs DIM Diff Check')
    failed_list.append('STG vs DIM Diff Check')

## Test 7 — Row Count Reconciliation: HZ vs DWH

Compares filtered row counts between HZ and DWH layers to confirm data completeness.

In [ ]:
def row_count(filter_list, schema_name, table_name):
    """
    Return the row count from a table, with optional WHERE filters.

    Args:
        filter_list: List of [column_name, operator, value] triples, e.g.
                     [['status', '=', 'ACTIVE'], ['date_code', '<=', '2024-12-31']]
        schema_name: Schema of the target table.
        table_name:  Name of the target table.
    """
    if filter_list:
        conditions = [f" {col} {op} '{val}' " for col, op, val in filter_list]
        query = f'''
            SELECT COUNT(*)
            FROM {schema_name}.{table_name}
            WHERE {' AND '.join(conditions)}
        '''
    else:
        query = f'SELECT COUNT(*) FROM {schema_name}.{table_name}'

    cursor.execute(query)
    return cursor.fetchall()

try:
    # Example filter: [column_name, operator, value]
    filter_list_dwh = []   # e.g. [['status', '=', 'ACTIVE']]
    filter_list_hz  = []

    count_dwh = row_count(filter_list_dwh, dwh_schema_name, dwh_table_name)
    count_hz  = row_count(filter_list_hz,  hz_schema_name,  hz_table_name)

    assert count_dwh == count_hz
    print(f'PASS  HZ vs DWH Count — DWH: {count_dwh[0][0]}  HZ: {count_hz[0][0]}')
    passed_list.append('HZ vs DWH Row Count')
except Exception:
    print(f'FAIL  HZ vs DWH Count — DWH: {count_dwh}  HZ: {count_hz}')
    failed_list.append('HZ vs DWH Row Count')

## Test Summary

In [ ]:
print('=' * 55)
print(f'  RESULTS: {len(passed_list)} passed   {len(failed_list)} failed')
print('=' * 55)

if passed_list:
    print('\nPASSED:')
    for i, name in enumerate(passed_list, 1):
        print(f'  {i}. {name}')

if failed_list:
    print('\nFAILED:')
    for i, name in enumerate(failed_list, 1):
        print(f'  {i}. {name}')
print()